In [1]:
!unzip ed_es_vtp_meshes.zip

Archive:  ed_es_vtp_meshes.zip
   creating: es_vtp_meshes_smoothed/
   creating: es_vtp_meshes_smoothed/content/
   creating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/
   creating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient101/
  inflating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient101/patient101_LV.vtp  
  inflating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient101/patient101_merged.vtp  
  inflating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient101/patient101_MYO.vtp  
  inflating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient101/patient101_RV.vtp  
   creating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient102/
  inflating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient102/patient102_LV.vtp  
  inflating: es_vtp_meshes_smoothed/content/vtp_meshes_smoothed_all_patients/patient10

In [3]:
!rm -rf es_vtp_meshes_smoothed/

In [2]:
# -----------------------------
# Full memory-efficient Colab script
# -----------------------------
# Paste this entire cell into Google Colab and run.

# 0) Installs (quiet)
!pip install -q pyvista trimesh open3d pycpd imageio matplotlib scikit-learn torch torchvision tqdm

# 1) Imports
import os, sys, gc, math, random, time, glob
import numpy as np
import pyvista as pv
import trimesh
import open3d as o3d
from pycpd import DeformableRegistration
from sklearn.neighbors import NearestNeighbors
import imageio, matplotlib.pyplot as plt
from tqdm import tqdm
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 2) User config
ED_DIR = "/content/ed_vtp_meshes"
ES_DIR = "/content/es_vtp_meshes"
CORR_DIR = "/content/corresponded"
os.makedirs(CORR_DIR, exist_ok=True)
MAX_VERTS_DECIMATE = 2500   # reduce if OOM (1000-3000 reasonable)
USE_NONRIGID_REGISTRATION = False  # set True only for a few patients if you need higher quality (costly)
ASSUME_CORRESPONDENCE = False  # set True if ED/ES already have identical vertex indices per patient
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)




In [3]:

# 3) Helpers: read VTP, normalize, decimate, ICP, CPD fallback
def read_vtp(path):
    m = pv.read(path)
    pts = m.points.copy()
    faces = None
    if m.faces is not None and len(m.faces)>0:
        arr = m.faces
        # convert pyvista face array to Nx3 faces if possible
        faces_list = []
        i = 0
        while i < len(arr):
            n = int(arr[i]); i+=1
            if n >= 3:
                face = [int(arr[j]) for j in range(i, i+3)]
                faces_list.append(face)
            i += n
        if len(faces_list)>0:
            faces = np.array(faces_list, dtype=np.int64)
    return pts, faces

def normalize_points(pts):
    c = pts.mean(axis=0)
    ptsc = pts - c
    s = max(1e-8, np.max(np.linalg.norm(ptsc, axis=1)))
    return ptsc / s, c, s

def decimate_mesh_if_needed(verts, faces, max_verts=2500):
    # If faces is None, just sample points
    if faces is None:
        if verts.shape[0] <= max_verts:
            return verts.copy(), None
        idx = np.random.choice(verts.shape[0], max_verts, replace=False)
        return verts[idx].copy(), None
    # build pyvista mesh
    faces_flat = np.hstack([np.full((faces.shape[0],1),3), faces]).astype(np.int64)
    mesh = pv.PolyData(verts, faces_flat)
    if mesh.n_points <= max_verts:
        return mesh.points.copy(), faces
    frac = max(0.01, float(max_verts) / float(mesh.n_points))
    try:
        dec = mesh.decimate_pro(frac, preserve_topology=True)
        f = dec.faces.reshape(-1,4)[:,1:4].astype(np.int64)
        return dec.points.copy(), f
    except Exception as e:
        # fallback sample surface points
        samp = mesh.sample_points(max_verts).points
        return samp.copy(), None

def rigid_icp_align(src_pts, tgt_pts, max_iter=30):
    src = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(src_pts.astype(np.float64)))
    tgt = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(tgt_pts.astype(np.float64)))
    try:
        res = o3d.pipelines.registration.registration_icp(
            src, tgt, 0.05, np.eye(4),
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=max_iter)
        )
        src.transform(res.transformation)
        return np.asarray(src.points), res.transformation
    except Exception as e:
        return src_pts.copy(), np.eye(4)

def safe_nonrigid_cpd_map(template_pts, target_pts, max_iter=30):
    # reduce sizes for CPD to save memory
    tpl = np.asarray(template_pts, dtype=np.float64)
    tgt = np.asarray(target_pts, dtype=np.float64)
    if tpl.shape[0] > 2000:
        idx = np.random.choice(tpl.shape[0], 2000, replace=False)
        tpl_small = tpl[idx]
    else:
        tpl_small = tpl
    if tgt.shape[0] > 2000:
        idx = np.random.choice(tgt.shape[0], 2000, replace=False)
        tgt_small = tgt[idx]
    else:
        tgt_small = tgt
    reg = DeformableRegistration(X=tgt_small, Y=tpl_small, max_iterations=max_iter, beta=2.0, lamb=3.0)
    TY, _ = reg.register()  # TY corresponds to tpl_small order
    # map full template points to nearest transformed sampled tpl points
    nbr = NearestNeighbors(n_neighbors=1).fit(tpl_small)
    _, idx_map = nbr.kneighbors(tpl)
    TY = np.asarray(TY)
    TY_full = TY[idx_map[:,0]]
    return TY_full


In [4]:
# 4) Discover patients
def discover_patients(ed_dir, es_dir):
    ed_dirs = sorted([d for d in os.listdir(ed_dir) if os.path.isdir(os.path.join(ed_dir,d))])
    es_dirs = sorted([d for d in os.listdir(es_dir) if os.path.isdir(os.path.join(es_dir,d))])
    common = [p for p in ed_dirs if p in es_dirs]
    pairs = []
    for p in common:
        edp = os.path.join(ed_dir, p, f"{p}_merged.vtp")
        esp = os.path.join(es_dir, p, f"{p}_merged.vtp")
        if os.path.exists(edp) and os.path.exists(esp):
            pairs.append((p, edp, esp))
    print(f"Found {len(pairs)} paired patients.")
    return pairs

pairs = discover_patients(ED_DIR, ES_DIR)
if len(pairs) == 0:
    raise RuntimeError("No patient pairs found. Check folder structure and filenames.")

# 5) Select template (first ED) and prepare template (decimate if needed)
tpl_id, tpl_ed_path, _ = pairs[0]
tpl_raw, tpl_faces = read_vtp(tpl_ed_path)
tpl_norm, tpl_center, tpl_scale = normalize_points(tpl_raw)
tpl_pts, tpl_faces_dec = decimate_mesh_if_needed(tpl_norm, tpl_faces, max_verts=MAX_VERTS_DECIMATE)
faces_global = tpl_faces_dec  # may be None if decimated to points; adjacency requires faces
if faces_global is None:
    raise RuntimeError("Template decimation removed faces; choose higher max_verts or use meshes with faces.")

Found 50 paired patients.


In [5]:

# 6) One-by-one processing -> save compact .npz files
def process_and_save(patient_id, ed_path, es_path, out_dir, template_pts, template_faces, max_verts=2500, use_nonrigid=False):
    print("Processing", patient_id)
    ed_raw, ed_faces = read_vtp(ed_path)
    es_raw, es_faces = read_vtp(es_path)
    # normalize by ED centroid/scale
    edn, c, s = normalize_points(ed_raw)
    esn = (es_raw - c) / s
    # decimate ED/ES for registration speed
    edd_pts, edd_faces = decimate_mesh_if_needed(edn, ed_faces, max_verts=max_verts)
    esd_pts, esd_faces = decimate_mesh_if_needed(esn, es_faces, max_verts=max_verts)
    # rigid align ed decimated to template frame
    ed_aligned, _ = rigid_icp_align(edd_pts, template_pts)
    # try nonrigid CPD if enabled (costly); otherwise use template directly
    if use_nonrigid:
        try:
            ed_corr = safe_nonrigid_cpd_map(template_pts, ed_aligned, max_iter=30)
        except Exception as e:
            print("CPD failed for", patient_id, ":", e)
            ed_corr = template_pts.copy()
    else:
        ed_corr = template_pts.copy()
    # align ES decimated to ED_aligned and map via NN to ed_corr
    es_aligned, _ = rigid_icp_align(esd_pts, ed_aligned)
    nbr = NearestNeighbors(n_neighbors=1).fit(es_aligned)
    dists, idxs = nbr.kneighbors(ed_corr)
    es_corr = es_aligned[idxs[:,0]]
    # Save compressed .npz (float32)
    outp = os.path.join(out_dir, f"{patient_id}.npz")
    np.savez_compressed(outp, ed=ed_corr.astype(np.float32), es=es_corr.astype(np.float32))
    # free memory
    del ed_raw, ed_faces, es_raw, es_faces, edn, esn, edd_pts, edd_faces, esd_pts, esd_faces, ed_aligned, es_aligned
    gc.collect()
    return outp

# Only create .npz if not already present
saved = []
for pid, edp, esp in tqdm(pairs):
    outp = os.path.join(CORR_DIR, f"{pid}.npz")
    if os.path.exists(outp):
        saved.append(outp); continue
    try:
        out = process_and_save(pid, edp, esp, CORR_DIR, tpl_pts, tpl_faces_dec, max_verts=MAX_VERTS_DECIMATE, use_nonrigid=USE_NONRIGID_REGISTRATION)
        saved.append(out)
    except Exception as e:
        print("Failed:", pid, e)
    gc.collect(); torch.cuda.empty_cache()

print(f"Saved corresponded .npz files: {len(saved)}")

100%|██████████| 50/50 [00:00<00:00, 98365.48it/s]

Saved corresponded .npz files: 50


In [6]:

# 7) Build adjacency (1-ring) from faces_global
def build_adjacency(nv, faces):
    neighbors = [set() for _ in range(nv)]
    for f in faces:
        if len(f) < 3: continue
        i,j,k = int(f[0]), int(f[1]), int(f[2])
        neighbors[i].update([j,k]); neighbors[j].update([i,k]); neighbors[k].update([i,j])
    maxdeg = max(len(n) for n in neighbors)
    neigh_idx = -np.ones((nv, maxdeg), dtype=np.int64)
    neigh_mask = np.zeros((nv, maxdeg), dtype=np.float32)
    for i, nset in enumerate(neighbors):
        if len(nset)==0: continue
        arr = np.array(sorted(list(nset)), dtype=np.int64)
        neigh_idx[i, :len(arr)] = arr
        neigh_mask[i, :len(arr)] = 1.0
    return neigh_idx, neigh_mask

# load one npz to get vertex count
sample_npz = saved[0]
tmp = np.load(sample_npz)
Nverts = tmp['ed'].shape[0]; tmp.close()
neigh_idx, neigh_mask = build_adjacency(Nverts, faces_global)
if neigh_idx is None:
    raise RuntimeError("Failed to build adjacency.")
neigh_idx_t = torch.from_numpy(neigh_idx).long().to(DEVICE)
neigh_mask_t = torch.from_numpy(neigh_mask).to(DEVICE)


In [7]:

# 8) Streaming Dataset that loads .npz on demand
class CorrespDataset(Dataset):
    def __init__(self, npz_paths):
        self.paths = list(npz_paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        d = np.load(self.paths[idx])
        ed = d['ed'].astype(np.float32)
        es = d['es'].astype(np.float32)
        d.close()
        return {'ed': torch.from_numpy(ed), 'es': torch.from_numpy(es), 'id': os.path.basename(self.paths[idx])}

all_npz = sorted(glob.glob(os.path.join(CORR_DIR, "*.npz")))
random.shuffle(all_npz)
split = int(0.8 * len(all_npz))
train_paths = all_npz[:split]; val_paths = all_npz[split:]
train_ds = CorrespDataset(train_paths); val_ds = CorrespDataset(val_paths)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

In [8]:


# 9) Define model (lightweight MeshGNN)
class MeshGNN(nn.Module):
    def __init__(self, latent=16):
        super().__init__()
        self.latent = latent
        self.in_mlp = nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,latent), nn.ReLU())
        self.msg = nn.Sequential(nn.Linear(latent*2 + 3,128), nn.ReLU(), nn.Linear(128,128), nn.ReLU())
        self.out = nn.Linear(128, 3)
    def forward(self, pos, neigh_idx, neigh_mask):
        h = self.in_mlp(pos)
        idx = neigh_idx
        nbr_h = h[idx]
        nbr_pos = pos[idx]
        mask = neigh_mask.unsqueeze(-1)
        maskf = mask.float()
        denom = maskf.sum(dim=1).clamp(min=1.0)
        mean_h = (nbr_h * maskf).sum(dim=1) / denom
        mean_pos = (nbr_pos * maskf).sum(dim=1) / denom
        rel = mean_pos - pos
        inp = torch.cat([h, mean_h, rel], dim=-1)
        m = self.msg(inp)
        disp = self.out(m)
        return disp

# Laplacian loss helper
def laplacian_loss(pred_pos, neigh_idx, neigh_mask):
    nbr = pred_pos[neigh_idx]
    mask = neigh_mask.unsqueeze(-1)
    denom = mask.sum(dim=1).clamp(min=1.0)
    mean_nbr = (nbr * mask).sum(dim=1) / denom
    return F.mse_loss(pred_pos, mean_nbr)

# volume via trimesh (CPU)
def compute_volume(verts, faces):
    tm = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    return abs(float(tm.volume))

# augmentation (CPU)
def augment_pointcloud_cpu(ed_np):
    p = ed_np.copy()
    # rotation
    angles = np.deg2rad(np.random.uniform(-6,6,3))
    Rx = np.array([[1,0,0],[0,math.cos(angles[0]),-math.sin(angles[0])],[0,math.sin(angles[0]),math.cos(angles[0])]])
    Ry = np.array([[math.cos(angles[1]),0,math.sin(angles[1])],[0,1,0],[-math.sin(angles[1]),0,math.cos(angles[1])]])
    Rz = np.array([[math.cos(angles[2]),-math.sin(angles[2]),0],[math.sin(angles[2]),math.cos(angles[2]),0],[0,0,1]])
    R = Rz @ Ry @ Rx
    p = p @ R.T
    p = p * (1.0 + np.random.normal(0.0, 0.01))
    p += np.random.normal(0.0, 0.0008, p.shape)
    return p

# 10) Training (streaming, memory-conscious)
model = MeshGNN(latent=16).to(DEVICE)
opt = optim.Adam(model.parameters(), lr=1e-3)
sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5)
EPOCHS = 150
L_LAP = 0.01
L_VOL = 0.5
best_val = 1e9
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        ed = batch['ed'][0].numpy()  # CPU numpy
        es = batch['es'][0].to(DEVICE)
        ed_aug = augment_pointcloud_cpu(ed)
        ed_t = torch.from_numpy(ed_aug).float().to(DEVICE)
        opt.zero_grad()
        disp = model(ed_t, neigh_idx_t, neigh_mask_t)
        pred = ed_t + disp
        lpos = F.mse_loss(pred, es)
        llap = laplacian_loss(pred, neigh_idx_t, neigh_mask_t)
        # volume loss computed on CPU to avoid keeping big arrays on GPU
        try:
            pred_cpu = pred.detach().cpu().numpy()
            vol_pred = compute_volume(pred_cpu, faces_global)
            vol_gt = compute_volume(batch['es'][0].numpy(), faces_global)
            lvol = torch.tensor((vol_pred - vol_gt)**2, device=DEVICE)
        except Exception:
            lvol = torch.tensor(0.0, device=DEVICE)
        loss = lpos + L_LAP * llap + L_VOL * lvol
        loss.backward()
        opt.step()
        train_loss += loss.item()
        # cleanup GPU intermediates
        del ed_t, disp, pred, lpos, llap, lvol
        torch.cuda.empty_cache()
    # validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            ed = batch['ed'][0].numpy()
            es = batch['es'][0].to(DEVICE)
            ed_t = torch.from_numpy(ed).float().to(DEVICE)
            disp = model(ed_t, neigh_idx_t, neigh_mask_t)
            pred = ed_t + disp
            lpos = F.mse_loss(pred, es)
            llap = laplacian_loss(pred, neigh_idx_t, neigh_mask_t)
            try:
                pred_cpu = pred.cpu().numpy()
                vol_pred = compute_volume(pred_cpu, faces_global)
                vol_gt = compute_volume(batch['es'][0].numpy(), faces_global)
                lvol = torch.tensor((vol_pred - vol_gt)**2, device=DEVICE)
            except Exception:
                lvol = torch.tensor(0.0, device=DEVICE)
            loss = lpos + L_LAP * llap + L_VOL * lvol
            val_loss += loss.item()
            del ed_t, disp, pred, lpos, llap, lvol
            torch.cuda.empty_cache()
    train_loss /= max(1, len(train_loader))
    val_loss /= max(1, len(val_loader))
    sched.step(val_loss)
    print(f"Epoch {epoch}/{EPOCHS} train={train_loss:.6f} val={val_loss:.6f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model': model.state_dict()}, "/content/best_meshgnn.pt")
    gc.collect(); torch.cuda.empty_cache()


Epoch 1/150 train=0.183499 val=0.059652
Epoch 2/150 train=0.052097 val=0.021407
Epoch 3/150 train=0.047508 val=0.026518
Epoch 4/150 train=0.037129 val=0.022188
Epoch 5/150 train=0.038806 val=0.018067
Epoch 6/150 train=0.037211 val=0.008980
Epoch 7/150 train=0.031797 val=0.021286
Epoch 8/150 train=0.032785 val=0.021178
Epoch 9/150 train=0.032629 val=0.013447
Epoch 10/150 train=0.039282 val=0.012033
Epoch 11/150 train=0.032458 val=0.010174
Epoch 12/150 train=0.028708 val=0.026652
Epoch 13/150 train=0.033509 val=0.012965
Epoch 14/150 train=0.030742 val=0.014062
Epoch 15/150 train=0.039736 val=0.024439
Epoch 16/150 train=0.033835 val=0.016452
Epoch 17/150 train=0.028141 val=0.022541
Epoch 18/150 train=0.033719 val=0.018566
Epoch 19/150 train=0.031181 val=0.015032
Epoch 20/150 train=0.032039 val=0.016962
Epoch 21/150 train=0.029340 val=0.008757
Epoch 22/150 train=0.028188 val=0.014969
Epoch 23/150 train=0.029494 val=0.009113
Epoch 24/150 train=0.032035 val=0.009439
Epoch 25/150 train=0.0292

In [9]:


# 11) Evaluation on validation set: RMSE and EF
def vertex_rmse(a, b):
    return np.sqrt(np.mean(np.sum((a-b)**2, axis=1)))

model.eval()
results = []
with torch.no_grad():
    for batch in val_loader:
        ed_np = batch['ed'][0].numpy()
        es_np = batch['es'][0].numpy()
        ed_t = torch.from_numpy(ed_np).float().to(DEVICE)
        disp = model(ed_t, neigh_idx_t, neigh_mask_t).cpu().numpy()
        pred = ed_np + disp
        rmse = vertex_rmse(pred, es_np)
        try:
            vol_pred = compute_volume(pred, faces_global)
            vol_gt = compute_volume(es_np, faces_global)
            ef_pred = (compute_volume(ed_np, faces_global) - vol_pred) / max(1e-8, compute_volume(ed_np, faces_global))
            ef_gt = (compute_volume(ed_np, faces_global) - vol_gt) / max(1e-8, compute_volume(ed_np, faces_global))
        except Exception:
            vol_pred = vol_gt = ef_pred = ef_gt = np.nan
        results.append({'rmse': rmse, 'ef_gt': ef_gt, 'ef_pred': ef_pred})
        del ed_t, disp
        gc.collect()
print("Validation results (summary):", np.mean([r['rmse'] for r in results]), "rmse mean")

# 12) Visualization: animate first saved patient (ED -> predicted ES)
vis_pngs = []
if len(saved)>0:
    sample = saved[0]
    d = np.load(sample)
    ed = d['ed']; es = d['es']; d.close()
    ed_t = torch.from_numpy(ed).float().to(DEVICE)
    with torch.no_grad():
        disp = model(ed_t, neigh_idx_t, neigh_mask_t).cpu().numpy()
    # create frames by scaling displacement (ease-in-out)
    nframes = 30
    frames = []
    pv.set_plot_theme("document")
    plotter = pv.Plotter(off_screen=True, window_size=(600,600))
    for fidx in range(nframes+1):
        alpha = 0.5 - 0.5*math.cos(math.pi * fidx / nframes)
        verts = ed + alpha * disp
        mesh = pv.PolyData(verts, np.hstack([np.full((faces_global.shape[0],1),3), faces_global]).astype(np.int64))
        plotter.clear()
        plotter.add_mesh(mesh, scalars=np.linalg.norm(disp, axis=1), show_scalar_bar=True)
        plotter.add_text(f"frame {fidx}", font_size=12)
        img = plotter.screenshot(None)
        frames.append(img)
    plotter.close()
    out_gif = "/content/ed_to_es_animation.gif"
    imageio.mimsave(out_gif, frames, fps=20)
    print("Saved animation to", out_gif)
else:
    print("No saved correspondences available for visualization.")

print("Done. Files saved in /content: corresponded npz files, /content/best_meshgnn.pt, animation gif (if created).")
print("Tips: If you still hit OOM, lower MAX_VERTS_DECIMATE to 1000 and reduce EPOCHS or model latent dim.")


Validation results (summary): 0.09163634 rmse mean
Saved animation to /content/ed_to_es_animation.gif
Done. Files saved in /content: corresponded npz files, /content/best_meshgnn.pt, animation gif (if created).
Tips: If you still hit OOM, lower MAX_VERTS_DECIMATE to 1000 and reduce EPOCHS or model latent dim.
